# Mask Methods Deep Dive

The ocean/land mask is the most consequential decision in the bathymetry pipeline. A wrong mask means wrong boundary conditions for the entire ocean simulation.

`mom6_forge` provides three mask methods, each with different accuracy/cost trade-offs:

| Method | Function | Source | Cost |
|---|---|---|---|
| `ocean_frac` | `generate_mask_ocean_frac` | Source bathymetry (sub-sampling) | Medium |
| `cartopy` | `generate_mask_cartopy` | Natural Earth land polygons | Low |
| `from_processed_depth` | `generate_mask_from_processed_depth` | Sign of regridded depth | Negligible |

This notebook visualises what each one does internally and compares their outputs on the same toy grid.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
import tempfile

from mom6_forge.grid import Grid
from mom6_forge.topo import Topo
from mom6_forge._source_bathy import SourceBathy

plt.rcParams["figure.dpi"] = 120

## Setup — same toy grid as notebook 8

In [ ]:
grid = Grid(resolution=2.0, xstart=262.0, lenx=10.0, ystart=20.0, leny=8.0, name="toy")

src_lons = np.arange(260.0, 274.1, 0.1)
src_lats = np.arange(18.0,  30.1, 0.1)
src_lon2d, src_lat2d = np.meshgrid(src_lons, src_lats)

elevation = np.full(src_lon2d.shape, -2000.0)
elevation[(src_lat2d > 26.0) & (src_lon2d < 265.0)] = +50.0   # NW land peninsula
elevation[
    (src_lat2d > 22.0) & (src_lat2d < 24.0) &
    (src_lon2d > 266.0) & (src_lon2d < 268.0)
] = -80.0  # shallow bank

tmp_dir = Path(tempfile.mkdtemp())
src_path = tmp_dir / "synthetic_bathy.nc"
xr.Dataset(
    {"elevation": (["lat", "lon"], elevation.astype("float32"))},
    coords={"lon": src_lons, "lat": src_lats},
).to_netcdf(src_path)

topo = Topo(grid, min_depth=5.0, version_control_dir=tmp_dir)
topo.set_flat(1000)
src = SourceBathy(src_path).slice_to_domain(topo)

print("Setup complete.")

## Method 1: `ocean_frac` — Monte-Carlo sub-sampling

This mirrors the Fortran program `create_model_topo.f90` from the tx2_3 topography workflow.

### How it works

For each model T-cell:
1. Distribute `nx_sub × ny_sub` interior sub-points evenly across the cell using **bilinear interpolation** of the 4 Q-point corners
2. Snap each sub-point to the nearest source pixel
3. Count how many sub-points land on ocean source pixels → `OCN_FRAC`
4. If `OCN_FRAC ≥ mask_threshold` → ocean, else → land

The sub-points are always strictly interior (never on cell edges) because the fraction is computed as `k/(n+1)` for k=1..n.

In [ ]:
# Visualise where sub-points fall for one specific cell
def subpoints_for_cell(grid, j, i, nx_sub, ny_sub):
    """Return (lon, lat) of nx_sub*ny_sub sub-points for T-cell (j, i)."""
    SW_lon = grid.qlon.values[j,   i  ]
    SE_lon = grid.qlon.values[j,   i+1]
    NE_lon = grid.qlon.values[j+1, i+1]
    NW_lon = grid.qlon.values[j+1, i  ]
    SW_lat = grid.qlat.values[j,   i  ]
    SE_lat = grid.qlat.values[j,   i+1]
    NE_lat = grid.qlat.values[j+1, i+1]
    NW_lat = grid.qlat.values[j+1, i  ]

    pts_lon, pts_lat = [], []
    for js in range(1, ny_sub + 1):
        jf = js / (ny_sub + 1)
        for is_ in range(1, nx_sub + 1):
            iff = is_ / (nx_sub + 1)
            lon = ((1-iff)*(1-jf)*SW_lon + iff*(1-jf)*SE_lon
                   + iff*jf*NE_lon + (1-iff)*jf*NW_lon)
            lat = ((1-iff)*(1-jf)*SW_lat + iff*(1-jf)*SE_lat
                   + iff*jf*NE_lat + (1-iff)*jf*NW_lat)
            pts_lon.append(lon)
            pts_lat.append(lat)
    return np.array(pts_lon), np.array(pts_lat)

nx_sub, ny_sub = 5, 5

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax_idx, (jc, ic) in enumerate([(3, 0), (1, 2)]):
    ax = axes[ax_idx]

    # Source elevation background
    ax.pcolormesh(
        src_lons, src_lats, elevation,
        cmap="RdBu", vmin=-2200, vmax=200, shading="auto", alpha=0.6
    )

    # All model cell outlines
    ax.pcolormesh(
        grid.qlon.values, grid.qlat.values,
        np.zeros((grid.ny, grid.nx)),
        edgecolors="k", linewidth=0.8, facecolor="none", shading="flat"
    )

    # Sub-points for this cell
    sub_lon, sub_lat = subpoints_for_cell(grid, jc, ic, nx_sub, ny_sub)

    # Determine which sub-points are ocean in source
    dlon = float(src_lons[1] - src_lons[0])
    dlat = float(src_lats[1] - src_lats[0])
    ii = np.round((sub_lon - src_lons[0]) / dlon).astype(int).clip(0, len(src_lons)-1)
    jj = np.round((sub_lat - src_lats[0]) / dlat).astype(int).clip(0, len(src_lats)-1)
    is_ocean_sub = elevation[jj, ii] < 0

    ocn_frac = is_ocean_sub.mean()

    ax.scatter(sub_lon[is_ocean_sub],  sub_lat[is_ocean_sub],
               c="royalblue", s=60, zorder=6, label="Ocean sub-point")
    ax.scatter(sub_lon[~is_ocean_sub], sub_lat[~is_ocean_sub],
               c="saddlebrown", s=60, zorder=6, marker="x", label="Land sub-point")

    # T-cell centre
    ax.scatter(grid.tlon.values[jc, ic], grid.tlat.values[jc, ic],
               c="yellow", s=100, zorder=7, edgecolors="k", label="T-cell centre")

    ax.set_xlim(260.5, 273.5)
    ax.set_ylim(18.5, 29.5)
    ax.set_title(f"Cell ({jc},{ic}) — OCN_FRAC = {ocn_frac:.2f} → {'OCEAN' if ocn_frac >= 0.5 else 'LAND'}")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.legend(fontsize=8, loc="upper right")

plt.suptitle(f"ocean_frac sub-sampling: {nx_sub}×{ny_sub} = {nx_sub*ny_sub} sub-points per cell", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Compute mask for the whole grid and show OCN_FRAC
mask_of = topo.generate_mask_ocean_frac(src, nx_sub=nx_sub, ny_sub=ny_sub, mask_threshold=0.5)
ocn_frac_grid = src._topo_stats["OCN_FRAC"].values

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# OCN_FRAC values
ax = axes[0]
c = ax.pcolormesh(grid.qlon.values, grid.qlat.values, ocn_frac_grid,
                  cmap="Blues", vmin=0, vmax=1, shading="flat")
plt.colorbar(c, ax=ax, label="OCN_FRAC")
for j in range(grid.ny):
    for i in range(grid.nx):
        ax.text(grid.tlon.values[j,i], grid.tlat.values[j,i],
                f"{ocn_frac_grid[j,i]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("OCN_FRAC (raw fraction)")

# Threshold effect — show what changes at different thresholds
for ax, thresh, col in zip(axes[1:], [0.3, 0.7], [1, 2]):
    mask_t = (ocn_frac_grid >= thresh).astype(int)
    c = ax.pcolormesh(grid.qlon.values, grid.qlat.values, mask_t,
                      cmap="RdBu", vmin=0, vmax=1, shading="flat")
    plt.colorbar(c, ax=ax)
    for j in range(grid.ny):
        for i in range(grid.nx):
            label = "OCN" if mask_t[j, i] else "LND"
            ax.text(grid.tlon.values[j,i], grid.tlat.values[j,i],
                    label, ha="center", va="center", fontsize=8)
    ax.set_title(f"mask_threshold = {thresh}")
    for ax_ in axes:
        ax_.set_xlabel("Longitude")
        ax_.set_ylabel("Latitude")

plt.suptitle("Effect of mask_threshold on ocean_frac mask", fontsize=11)
plt.tight_layout()
plt.show()

print("Threshold 0.3:", int((ocn_frac_grid >= 0.3).sum()), "ocean cells")
print("Threshold 0.5:", int((ocn_frac_grid >= 0.5).sum()), "ocean cells")
print("Threshold 0.7:", int((ocn_frac_grid >= 0.7).sum()), "ocean cells")

### Depth statistics cached on `src`

The `ocean_frac` method caches per-cell depth statistics on `src._topo_stats`. These are used later by `write_topo` to add the `h2` (topographic roughness) field needed for MOM6's topographic drag parameterisation.

In [ ]:
stats = src._topo_stats
print("Cached topo stats variables:", list(stats.data_vars))
print()
print("D_mean (mean ocean depth per cell, m):")
print(np.round(stats["D_mean"].values, 1))
print()
print("h² = D2_mean - D_mean² (topographic variance, m²):")
h2 = stats["D2_mean"].values - stats["D_mean"].values**2
print(np.round(h2, 1))

## Method 2: `cartopy` — Natural Earth land polygons

Rasterises Natural Earth land polygon shapefiles onto the model grid using Shapely. Each T-cell centre is tested for point-in-polygon against the land geometry. No sub-sampling, no source bathymetry needed.

**Faster** than `ocean_frac` but coarser — it only knows whether the T-cell *centre* is on land or ocean, not the sub-cell fraction. This makes it unreliable for cells that straddle a coastline.

In [ ]:
mask_cartopy = topo.generate_mask_cartopy(resolution="50m")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, mask, title in [
    (axes[0], mask_of.values,     "ocean_frac (threshold=0.5)"),
    (axes[1], mask_cartopy.values, "cartopy (Natural Earth 50m)"),
]:
    c = ax.pcolormesh(grid.qlon.values, grid.qlat.values, mask,
                      cmap="RdBu", vmin=0, vmax=1, shading="flat")
    plt.colorbar(c, ax=ax)
    for j in range(grid.ny):
        for i in range(grid.nx):
            label = "OCN" if mask[j, i] else "LND"
            ax.text(grid.tlon.values[j,i], grid.tlat.values[j,i],
                    label, ha="center", va="center", fontsize=9)
    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

plt.suptitle("Mask method comparison: ocean_frac vs cartopy", fontsize=11)
plt.tight_layout()
plt.show()

# Check agreement
agree = (mask_of.values == mask_cartopy.values)
print(f"Cells where both methods agree: {agree.sum()}/{agree.size}")
print(f"Cells that differ: {(~agree).sum()}")

## Method 3: `generate_mask_from_processed_depth`

This is the simplest method — used internally by `tidy_dataset` when no external mask is provided. It derives the mask from the **sign of the already-regridded depth field**:
- depth ≤ 0 → land
- depth > 0 → ocean

**Key constraint:** this only works after the sign convention has been corrected (`positive_down=True` or the sign flip applied). It's not a mask generation step in the same sense as the other two — it's just reading the mask implicit in the depth values.

This is appropriate for the **direct pipeline** where the xesmf regrid has already produced a depth field and you want `tidy_dataset` to clean it up without needing a pre-computed mask.

In [ ]:
# Simulate a regridded bathymetry dataset (positive-down)
ny, nx = grid.ny, grid.nx
depth_arr = np.full((ny, nx), 1500.0)
depth_arr[3, 0] = -10.0   # cell (3,0) is land in NW corner
depth_arr[3, 1] = -5.0    # cell (3,1) also land

bathy_ds = xr.Dataset(
    {"depth": (["ny", "nx"], depth_arr)},
    coords={
        "lon": (["ny", "nx"], grid.tlon.values),
        "lat": (["ny", "nx"], grid.tlat.values),
    },
)

mask_from_depth = topo.generate_mask_from_processed_depth(bathy_ds)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
c = ax.pcolormesh(grid.qlon.values, grid.qlat.values, depth_arr,
                  cmap="RdBu", vmin=-100, vmax=2000, shading="flat")
plt.colorbar(c, ax=ax, label="Depth (m, +down)")
for j in range(ny):
    for i in range(nx):
        ax.text(grid.tlon.values[j,i], grid.tlat.values[j,i],
                f"{depth_arr[j,i]:.0f}", ha="center", va="center", fontsize=8)
ax.set_title("Regridded depth (positive-down)")

ax = axes[1]
c = ax.pcolormesh(grid.qlon.values, grid.qlat.values, mask_from_depth.values,
                  cmap="RdBu", vmin=0, vmax=1, shading="flat")
plt.colorbar(c, ax=ax)
for j in range(ny):
    for i in range(nx):
        label = "OCN" if mask_from_depth.values[j, i] else "LND"
        ax.text(grid.tlon.values[j,i], grid.tlat.values[j,i],
                label, ha="center", va="center", fontsize=9)
ax.set_title("Mask from depth sign (depth > 0 = ocean)")

for ax in axes:
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

plt.suptitle("generate_mask_from_processed_depth", fontsize=11)
plt.tight_layout()
plt.show()

## All three masks side by side

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

masks = [
    (mask_of.values,          "ocean_frac\n(5×5 sub-points, threshold=0.5)"),
    (mask_cartopy.values,     "cartopy\n(Natural Earth 50m)"),
    (mask_from_depth.values,  "from_processed_depth\n(sign of regridded depth)"),
]

for ax, (mask, title) in zip(axes, masks):
    ax.pcolormesh(grid.qlon.values, grid.qlat.values, mask,
                  cmap="RdBu", vmin=0, vmax=1, shading="flat")
    for j in range(grid.ny):
        for i in range(grid.nx):
            label = "OCN" if mask[j, i] else "LND"
            color = "white" if mask[j, i] else "k"
            ax.text(grid.tlon.values[j,i], grid.tlat.values[j,i],
                    label, ha="center", va="center", fontsize=9, color=color)
    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

plt.suptitle("Mask method comparison — toy 5×4 grid", fontsize=12)
plt.tight_layout()
plt.show()

## `tidy_dataset` — what happens to the mask after generation

Regardless of which mask method is used, `tidy_dataset` always runs these cleanup steps on the mask before enforcing depths:

1. **`binary_fill_holes`** — any ocean region completely surrounded by land is filled in as land (inland lake removal)
2. **Single-cell channel fill** — ocean cells with land on both sides in either x or y direction are converted to land
3. **Optional diagonal channel fill** — same for diagonal connections (when `fill_channels=True`)

Let's visualise what `binary_fill_holes` does to a mask with an artificial inland lake.

In [ ]:
from scipy.ndimage import binary_fill_holes

# Artificial mask with an inland lake (ocean surrounded by land)
mask_with_lake = np.array([
    [1, 1, 1, 1, 1],
    [1, 0, 0, 0, 1],
    [1, 0, 1, 0, 1],   # <-- cell (2,2) is ocean, but landlocked
    [1, 0, 0, 0, 1],
], dtype=int)

land_mask = np.abs(mask_with_lake - 1)
land_filled = binary_fill_holes(land_mask).astype(int)
ocean_after_fill = np.abs(land_filled - 1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, data, title in [
    (axes[0], mask_with_lake, "Before: landlocked ocean cell at (2,2)"),
    (axes[1], ocean_after_fill, "After: binary_fill_holes removes inland lake"),
]:
    ax.imshow(data, cmap="RdBu", vmin=0, vmax=1, origin="lower", aspect="equal")
    for j in range(data.shape[0]):
        for i in range(data.shape[1]):
            label = "OCN" if data[j, i] else "LND"
            ax.text(i, j, label, ha="center", va="center", fontsize=9)
    ax.set_title(title)
    ax.set_xticks(range(data.shape[1]))
    ax.set_yticks(range(data.shape[0]))
    ax.set_xticklabels([f"i={k}" for k in range(data.shape[1])])
    ax.set_yticklabels([f"j={k}" for k in range(data.shape[0])])

plt.suptitle("tidy_dataset: lake removal via binary_fill_holes", fontsize=11)
plt.tight_layout()
plt.show()

## Summary

| Method | Uses source data | Sub-cell accuracy | Needs bathymetry loaded | Best used in |
|---|---|---|---|---|
| `ocean_frac` | Yes | Yes (sub-sampling) | Yes | High-res pipeline |
| `cartopy` | No | No (cell-centre only) | No | Quick check, or high-res pipeline |
| `from_processed_depth` | Implicit (regridded depth) | No | After regridding | Direct pipeline (auto) |

**Recommendation:** use `ocean_frac` for production workflows. Use `cartopy` for quick sanity checks or when source data is not yet loaded. `from_processed_depth` is the internal fallback.

**Next:** `10_cressman_interpolation.ipynb` — how the Cressman weights are computed and applied.